# Text Summarization Training
This notebook demonstrates how to load the dataset, train the Text Summarization model, and evaluate it.

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
from torch import optim
import mlflow
import os
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader, TensorDataset
import logging

# Ensure src module is visible
import sys
sys.path.append("..")
from src.summarization.encoder import EncoderBiLSTM
from src.summarization.decoder import AttnDecoderLSTM
from src.summarization.train import Seq2SeqSummarizer, train_epoch
from src.evaluation.rouge import SummarizationEvaluator
from src.summarization.beam_search import decode_beam_search

os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5001"
mlflow.set_tracking_uri("http://localhost:5001")
mlflow.set_experiment("Summarization_Model")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 2. Load and Prepare Dataset

In [ ]:
logger.info("Loading `OpenHust/vietnamese-summarization` Dataset from HuggingFace...")
dataset = load_dataset("OpenHust/vietnamese-summarization", split="train[:500]")
eval_dataset_hf = load_dataset("OpenHust/vietnamese-summarization", split="train[500:550]")

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")

def process_data(examples):
    src_encoded = tokenizer(examples["Document"], padding="max_length", max_length=256, truncation=True)
    trg_encoded = tokenizer(examples["Summary"], padding="max_length", max_length=64, truncation=True)
    return {"src_ids": src_encoded["input_ids"], "trg_ids": trg_encoded["input_ids"]}

logger.info("Mapping tokenization over datasets...")
dataset = dataset.map(process_data, batched=True, remove_columns=dataset.column_names)
eval_dataset_hf = eval_dataset_hf.map(process_data, batched=True, remove_columns=eval_dataset_hf.column_names)

dataset.set_format(type="torch", columns=["src_ids", "trg_ids"])
eval_dataset_hf.set_format(type="torch", columns=["src_ids", "trg_ids"])

train_loader = DataLoader(TensorDataset(dataset["src_ids"], dataset["trg_ids"]), batch_size=16, shuffle=True, num_workers=0)
eval_loader = DataLoader(TensorDataset(eval_dataset_hf["src_ids"], eval_dataset_hf["trg_ids"]), batch_size=1, shuffle=False)

print(f"Train batches: {len(train_loader)}, Eval batches: {len(eval_loader)}")

## 3. Train Model

In [ ]:
INPUT_DIM = 50000
OUTPUT_DIM = 50000
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 1
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

enc = EncoderBiLSTM(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = AttnDecoderLSTM(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
model = Seq2SeqSummarizer(enc, dec, device).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
evaluator = SummarizationEvaluator()

with mlflow.start_run():
    for epoch in range(3):
        model.train()
        epoch_loss = 0
        for batch_idx, (src, trg) in enumerate(train_loader):
            src = src.to(device)
            trg = trg.to(device)
            
            optimizer.zero_grad()
            output = model(src, trg)
            
            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)
            
            loss = criterion(output, trg)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
            
        avg_loss = epoch_loss / len(train_loader)
        mlflow.log_metric("train_loss", avg_loss, step=epoch)
        logger.info(f"Epoch {epoch + 1}/3: Avg Loss: {avg_loss:.4f}")

## 4. Evaluation & Testing

In [ ]:
# Evaluate the summarization model
model.eval()
predictions = []
references = []

with torch.no_grad():
    for src, trg in eval_loader:
        src = src.to(device)
        encoder_outputs, hidden, cell = model.encoder(src)
        
        sos_id = tokenizer.cls_token_id or tokenizer.bos_token_id or 0
        eos_id = tokenizer.sep_token_id or tokenizer.eos_token_id or 2
        
        decoded_ids = decode_beam_search(model.decoder, encoder_outputs, hidden, cell, sos_id, eos_id, max_len=64, beam_width=3, device=device, min_len=10)
        
        pred_str = tokenizer.decode(decoded_ids, skip_special_tokens=True)
        ref_str = tokenizer.decode(trg[0], skip_special_tokens=True)
        
        predictions.append(pred_str)
        references.append(ref_str)

scores = evaluator.compute_scores(predictions, references)
print(f"Validation Scores - ROUGE-L: {scores.get('rougeL', 0):.4f} | BLEU: {scores.get('sacrebleu', 0):.4f}")

# Example output
print(f"Reference: {references[0]}")
print(f"Prediction: {predictions[0]}")